<a href="https://colab.research.google.com/github/meghansn/mental-health-llm-pipeline/blob/main/Synthetic_Data_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from google.colab import drive

# 1. Mount Drive (if you haven't already in this session)
drive.mount('/content/drive')

# 2. Define the path
CACHE_DIR = "/content/drive/MyDrive/project_data"

# 3. Create the directory if it doesn't exist
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)
    print(f"✅ Created directory: {CACHE_DIR}")
else:
    print(f"📂 Directory already exists: {CACHE_DIR}")

# 4. Now update your CACHE_FILE path
CACHE_FILE = os.path.join(CACHE_DIR, "generation_checkpoint.json")
print(f"💾 Checkpoint will be saved to: {CACHE_FILE}")

Mounted at /content/drive
📂 Directory already exists: /content/drive/MyDrive/project_data
💾 Checkpoint will be saved to: /content/drive/MyDrive/project_data/generation_checkpoint.json


In [ ]:
# ==============================================================================
# 1. GITHUB INITIALIZATION
# ==============================================================================
import os
import shutil

# Reset to root
os.chdir('/content')
if os.path.exists('mental-health-llm-pipeline'):
    shutil.rmtree('mental-health-llm-pipeline')

# Config
USERNAME = ""
TOKEN = ""
REPO_NAME = "mental-health-llm-pipeline"

!git clone https://:@github.com/meghansn/mental-health-llm-pipeline.git
os.chdir("mental-health-llm-pipeline")
print(f"🚀 Repository ready at: {os.getcwd()}")

In [3]:
# ==============================================================================
# 2. AUTHENTICATE GOOGLE
# ==============================================================================
from google.colab import auth
auth.authenticate_user()
print("🚀 Authenticated with Google Cloud.")

🚀 Authenticated with Google Cloud.


In [ ]:
# ==============================================================================
# 3. GCP BIGQUERY INITIALIZATION
# ==============================================================================
from google.cloud import bigquery

PROJECT_ID = "mental-health-llm-pip" # Make sure this matches your project exactly
bq_client = bigquery.Client(project=PROJECT_ID)

print("Resetting 'patient_insights' dataset...")
bq_client.delete_dataset("patient_insights", delete_contents=True, not_found_ok=True)
bq_client.create_dataset("patient_insights")

# Define Schema
bq_client.query(f"""
    CREATE TABLE `{PROJECT_ID}.patient_insights.dim_patients` (
        patient_id STRING, age INT64, gender STRING, region STRING,
        intake_date DATE, profile STRING
    );
""").result()

bq_client.query(f"""
    CREATE TABLE `{PROJECT_ID}.patient_insights.fact_sessions` (
        session_id STRING, patient_id STRING, session_date DATE,
        raw_transcript_unredacted STRING, medication_name STRING,
        medication_rxnorm_code STRING, dosage_mg INT64
    );
""").result()

print("🚀 Schema ready.")

In [ ]:
# ==============================================================================
# Step 4: PHASE 1: GENERATE & SAVE PATIENT DIMENSION (GTA LOCALIZED)
# ==============================================================================
!pip install faker
import time
from faker import Faker
import random
import uuid

# Define constants FIRST
GTA_CITIES = ["Toronto", "Mississauga", "Brampton", "Markham", "Vaughan", "Richmond Hill", "Oakville", "Burlington"]
PROFILES = {
    "Depressive": {"symptoms": ["persistent sadness", "fatigue", "lack of interest"], "meds": [("Sertraline", "769747"), ("Escitalopram", "237646")]},
    "Anxiety": {"symptoms": ["excessive worry", "restlessness", "muscle tension"], "meds": [("Buspirone", "313498"), ("Lorazepam", "38634")]},
    "Bipolar": {"symptoms": ["mood swings", "impulsivity", "racing thoughts"], "meds": [("Lithium", "31299"), ("Lamotrigine", "116527")]}
}

print("⏳ Waiting for BigQuery schema to propagate and verifying table...")
time.sleep(5)

# Generate data
fake = Faker(['en_CA'])
patients = [{
    "patient_id": str(uuid.uuid4()),
    "age": random.randint(18, 75),
    "gender": random.choice(["Male", "Female"]),
    "region": random.choice(GTA_CITIES),
    "intake_date": fake.date_between(start_date='-2y', end_date='-1y').strftime('%Y-%m-%d'),
    "profile": random.choice(list(PROFILES.keys()))
} for _ in range(1000)]

# INGESTION
table_path = f"{PROJECT_ID}.patient_insights.dim_patients"
batch_size = 500

for i in range(0, len(patients), batch_size):
    batch = patients[i:i + batch_size]
    try:
        errors = bq_client.insert_rows_json(table_path, batch)
        if errors:
            print(f"❌ Errors in batch {i}: {errors}")
        else:
            print(f"✅ Ingested batch {i//batch_size + 1} of patients.")
    except Exception as e:
        print(f"❌ Critical Error on batch {i}: {e}")
        break

print(f"🚀 Successfully processed patient ingestion.")

In [ ]:
#THIS FUNCTION DOESN'T TAKE INTO ACCOUNT GENDER
def get_diagnostic_prompt(profile, symptoms, med_name, month, last_session_summary=None):
    # If this is month 1, start fresh. If month 2+, use last_session_summary to keep continuity.
    context = f"Previous Session Summary: {last_session_summary}" if last_session_summary else "This is the first intake session."

    # We provide the list, but the prompt now instructs the model to CHOOSE
    # based on the flow of the previous session.
    prompt = f"""
    [SYSTEM INSTRUCTION]
    You are a clinical psychiatrist. Your goal is to conduct a longitudinal therapy process.

    PATIENT PERSONA ({profile}):
    - Profile: {profile}.
    - Active Symptoms: {', '.join(symptoms)}.
    - Current Med: {med_name}.

    {context}

    - INSTRUCTION:
      1. Review the previous session summary.
      2. Decide naturally: Does the patient continue the previous topic, or does a new life event (mundane or crisis) arise?
      3. Maintain strict character consistency. The patient's history, personality, and tone MUST NOT change.
      4. Use natural, messy, conversational English.

    THERAPIST PERSONA:
    - Track progress against their {profile} baseline.
    - Ask at least one evidence-based question about the efficacy of {med_name}.

    FORMATTING RULES:
    - 50+ lines of dialogue.
    - Start with [START], end with [END].
    """
    return prompt

In [ ]:
#RUN THIS QUERY WITH GENDER RULES
def get_diagnostic_prompt(profile, symptoms, med_name, month, gender, last_session_summary=None):
    # Pass gender explicitly into the persona rules
    pronouns = "she/her" if gender == "Female" else "he/him"

    if last_session_summary:
        context = f"Previous Session Summary: {last_session_summary}\n- This is Month {month} of treatment. The patient has met you before."
    else:
        context = f"This is the absolute FIRST intake session. You and the patient have NEVER met before. Do not say 'welcome back' or 'nice to see you again'."

    prompt = f"""
    [SYSTEM INSTRUCTION]
    You are a clinical psychiatrist. Your goal is to conduct a longitudinal therapy process.

    PATIENT PERSONA ({profile}):
    - Profile: {profile}.
    - Gender: {gender} (Use {pronouns} pronouns exclusively when referencing the patient).
    - Active Symptoms: {', '.join(symptoms)}.
    - Current Med: {med_name}.

    {context}

    - INSTRUCTION:
      1. Review the context. If it's the first session, conduct an intake evaluation.
      2. Decide naturally: Does the patient continue the previous topic, or does a new life event arise?
      3. Maintain strict character consistency.
      4. Use natural, messy, conversational English.

    THERAPIST PERSONA:
    - Track progress against their {profile} baseline.
    - Ask at least one evidence-based question about the efficacy of {med_name}.

    FORMATTING RULES:
    - 50+ lines of dialogue.
    - Start with [START], end with [END].
    """
    return prompt

In [ ]:
# ==============================================================================
# CELL 2: OPTIMIZED RESUMABLE GENERATOR (MEMORY-ENABLED + BATCHED)
# ==============================================================================
import json, os, uuid, random, vertexai, sys
from datetime import timedelta, datetime
from vertexai.generative_models import GenerativeModel
from google.cloud import bigquery

# 1. INITIALIZE

PROJECT_ID = "mental-health-llm-pip" # Make sure this matches your project exactly
bq_client = bigquery.Client(project=PROJECT_ID)
print("🚀 Initializing Vertex AI...", flush=True)
vertexai.init(project=PROJECT_ID, location="global")
model = GenerativeModel("gemini-3.1-flash-lite")
FACT_SESSIONS_PATH = f"{PROJECT_ID}.patient_insights.fact_sessions"
CACHE_FILE = "generation_checkpoint.json"

# Define constants FIRST
GTA_CITIES = ["Toronto", "Mississauga", "Brampton", "Markham", "Vaughan", "Richmond Hill", "Oakville", "Burlington"]
PROFILES = {
    "Depressive": {"symptoms": ["persistent sadness", "fatigue", "lack of interest"], "meds": [("Sertraline", "769747"), ("Escitalopram", "237646")]},
    "Anxiety": {"symptoms": ["excessive worry", "restlessness", "muscle tension"], "meds": [("Buspirone", "313498"), ("Lorazepam", "38634")]},
    "Bipolar": {"symptoms": ["mood swings", "impulsivity", "racing thoughts"], "meds": [("Lithium", "31299"), ("Lamotrigine", "116527")]}
}

# Load checkpoint
finished_patients = []
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r') as f:
        finished_patients = json.load(f)
    print(f"📂 Loaded {len(finished_patients)} previously finished patients.", flush=True)

# 2. FETCH PATIENTS
print("🔍 Fetching patients from BigQuery...", flush=True)
#patients = [dict(row) for row in bq_client.query(f"SELECT * FROM `{PROJECT_ID}.patient_insights.dim_patients`").result()]
query = f"SELECT * FROM `{PROJECT_ID}.patient_insights.dim_patients` ORDER BY patient_id"
patients = [dict(row) for row in bq_client.query(query).result()]
print(f"✅ Found {len(patients)} patients in BigQuery.", flush=True)



# 3. GENERATION LOOP
# Source of Truth: Query BQ to see what is already there
query_done = f"SELECT DISTINCT patient_id FROM `{PROJECT_ID}.patient_insights.fact_sessions`"
already_done_ids = {row['patient_id'] for row in bq_client.query(query_done).result()}

# Filter the list
patients = [p for p in patients if p['patient_id'] not in already_done_ids]
print(f"✅ Filtered list: {len(patients)} patients remaining to process.")

batch_sessions = []
batch_patient_ids = []

for idx, p in enumerate(patients):
    print(f"--- Processing Patient {idx+1}/{len(patients)} ({p['patient_id'][:8]}) ---", flush=True)

    last_summary = None
    intake = p['intake_date']
    if isinstance(intake, str): intake = datetime.strptime(intake, '%Y-%m-%d').date()

    for month in range(1, 13):
        med_name, med_code = random.choice(PROFILES[p['profile']]["meds"])

        # Call model for transcript
        #update call below to include gender
        prompt = get_diagnostic_prompt(p['profile'], PROFILES[p['profile']]["symptoms"], med_name, month, last_summary)
        transcript = model.generate_content(prompt).text

        # Summary for memory
        summary_prompt = f"Summarize in two sentences focusing on life events and patient state: {transcript}"
        last_summary = model.generate_content(summary_prompt).text

        batch_sessions.append({
            "session_id": str(uuid.uuid4()), "patient_id": p['patient_id'],
            "session_date": (intake + timedelta(days=30 * (month-1))).strftime('%Y-%m-%d'),
            "raw_transcript_unredacted": transcript,
            "medication_name": med_name, "medication_rxnorm_code": med_code,
            "dosage_mg": random.choice([10, 20, 50, 100])
        })
        print(f"  > Month {month} complete.", flush=True)

    batch_patient_ids.append(p['patient_id'])

    # BATCHING: Save to BQ every 5 patients
    if len(batch_patient_ids) >= 5:
        print(f"💾 Attempting to save {len(batch_sessions)} rows to BigQuery...", flush=True)
        errors = bq_client.insert_rows_json(FACT_SESSIONS_PATH, batch_sessions)

        if errors == []:
            print(f"✅ Batch successfully pushed to BigQuery!", flush=True)
            batch_sessions = []
            batch_patient_ids = []
        else:
            print(f"❌ Errors encountered: {errors}", flush=True)

In [ ]:
# Run this once to rebuild your checkpoint file from BigQuery
query = f"SELECT DISTINCT patient_id FROM `{PROJECT_ID}.patient_insights.fact_sessions`"
existing_ids = [row['patient_id'] for row in bq_client.query(query).result()]

# Save these to your new persistent path
with open(CACHE_FILE, 'w') as f:
    json.dump(existing_ids, f)

print(f"✅ Checkpoint file rebuilt with {len(existing_ids)} patients.")

In [ ]:
import json

# Check what the script actually sees
with open(CACHE_FILE, 'r') as f:
    checkpoint_data = json.load(f)

print(f"DEBUG: Found {len(checkpoint_data)} IDs in the file.")
print(f"DEBUG: First ID in file: {checkpoint_data[0]}")

# Check the first patient in your fetched list
first_patient_id = patients[0]['patient_id']
print(f"DEBUG: First patient ID from BigQuery: {first_patient_id}")

# The moment of truth: Does the list contain the patient?
is_in_list = first_patient_id in checkpoint_data
print(f"DEBUG: Is the first patient in the checkpoint? {is_in_list}")

In [ ]:
# Run this to see why it isn't skipping
first_patient = patients[0]
print(f"The first patient in the list is: {first_patient['patient_id']}")
print(f"Is this ID in our loaded checkpoint list? {first_patient['patient_id'] in finished_patients}")

In [ ]:
# DIAGNOSTIC CELL: Run this by itself
import vertexai
from vertexai.generative_models import GenerativeModel

print("1. Initializing Vertex...")
vertexai.init(project=PROJECT_ID, location="global")
model = GenerativeModel("gemini-3.1-flash-lite")

print("2. Attempting a single tiny request...")
response = model.generate_content("Say 'Hello' if you are working.")
print(f"3. Response received: {response.text}")

In [ ]:
# Define the content
data_dictionary_content = """# Data Dictionary: Mental Health Synthetic Dataset

## 1. Table: `dim_patients`
| Column | Data Type | Description |
| :--- | :--- | :--- |
| `patient_id` | STRING | Unique UUID assigned to each patient. |
| `age` | INT64 | Patient age (18–75). |
| `gender` | STRING | Gender identity. |
| `region` | STRING | GTA location. |
| `intake_date` | DATE | Onboarding date. |
| `profile` | STRING | Clinical category (Depressive, Anxiety, or Bipolar). |

## 2. Table: `fact_sessions`
| Column | Data Type | Description |
| :--- | :--- | :--- |
| `session_id` | STRING | Unique UUID for the session. |
| `patient_id` | STRING | Foreign key to `dim_patients`. |
| `session_date` | DATE | Date of session. |
| `raw_transcript_unredacted` | STRING | 50+ line therapy dialogue. |
| `medication_name` | STRING | Name of medication. |
| `medication_rxnorm_code` | STRING | Standard RxNorm code. |
| `dosage_mg` | INT64 | Dosage in milligrams. |
"""

# Write it to your repo folder
with open("/content/mental-health-llm-pipeline/data_dictionary.md", "w") as f:
    f.write(data_dictionary_content)

print("✅ data_dictionary.md saved to your repo folder.")

In [ ]:
# Create the README
readme = """# Synthetic Clinical Longitudinal Dataset

## Project Goal
Generating a high-fidelity, longitudinal synthetic dataset for clinical ML benchmarking.

## Methodology
Uses Gemini 3.1 Flash-Lite via Vertex AI to generate 12-month synthetic therapy trajectories, stored directly in BigQuery.

## Features
- **Checkpointing:** Resumable execution via `generation_checkpoint.json`.
- **Scalability:** Optimized for streaming inserts into BigQuery.

## Future Roadmap
1. PHI Redaction (via Cloud DLP).
2. Sentiment & Topic Modeling (BERTopic).
3. Clinical Mapping (DSM-5/ICD-10 classification).
4. Predictive Regression analysis.
"""

with open("README.md", "w") as f:
    f.write(readme)

# Create the .gitignore to protect your secrets
with open(".gitignore", "w") as f:
    f.write(".ipynb_checkpoints/\ngeneration_checkpoint.json\n*.json\n__pycache__/\n")

print("Documentation created successfully.")

In [ ]:
!git config --global user.email ""
!git config --global user.name ""

In [ ]:
!ls -F

In [5]:
# ==============================================================================
# POST-PROCESSING CLEAN-UP PIPELINE (Deterministic Fixes)
# ==============================================================================
import re
from google.cloud import bigquery

PROJECT_ID = "mental-health-llm-pip"
bq_client = bigquery.Client(project=PROJECT_ID)

print("🔍 Fetching sessions and patient metadata from BigQuery...")

# Join tables to get the correct gender for every session row
query = f"""
    SELECT
        f.session_id,
        f.raw_transcript_unredacted,
        p.gender,
        ROW_NUMBER() OVER(PARTITION BY f.patient_id ORDER BY f.session_date ASC) as session_sequence
    FROM `{PROJECT_ID}.patient_insights.fact_sessions` f
    JOIN `{PROJECT_ID}.patient_insights.dim_patients` p
      ON f.patient_id = p.patient_id
"""

sessions_df = bq_client.query(query).to_dataframe()
print(f"✅ Loaded {len(sessions_df)} sessions for cleaning.")

def clean_transcript(row):
    text = row['raw_transcript_unredacted']
    gender = row['gender']
    seq = row['session_sequence']

    # ---- FIX 1: First Session Greeting ----
    if seq == 1:
        text = re.sub(r"(?i)\bnice to see you again\b", "nice to meet you", text)
        text = re.sub(r"(?i)\bwelcome back\b", "welcome to our first session", text)
        text = re.sub(r"(?i)\bgood to see you again\b", "nice to meet you", text)

    # ---- FIX 2: Pronoun Realignment for Female Patients ----
    if gender == "Female":
        text = re.sub(r"\bhe\b", "she", text)
        text = re.sub(r"\bHe\b", "She", text)
        text = re.sub(r"\bhim\b", "her", text)
        text = re.sub(r"\bHim\b", "Her", text)
        text = re.sub(r"\bhis\b", "her", text)
        text = re.sub(r"\bHis\b", "Her", text)

    return text

print("🧹 Applying data-cleaning transformations...")
sessions_df['cleaned_transcript'] = sessions_df.apply(clean_transcript, axis=1)

# ---- WRITE BACK TO BIGQUERY VIA STAGING TABLE ----
print("💾 Uploading cleaned text to a temporary staging table...")

# Isolate just the target keys and the fixed text
staging_df = sessions_df[['session_id', 'cleaned_transcript']]

# Configure the load job
staging_table_id = f"{PROJECT_ID}.patient_insights.tmp_cleaned_transcripts"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE", # Overwrite staging table if it exists
    schema=[
        bigquery.SchemaField("session_id", "STRING"),
        bigquery.SchemaField("cleaned_transcript", "STRING")
    ]
)

# Load the dataframe to BigQuery
load_job = bq_client.load_table_from_dataframe(staging_df, staging_table_id, job_config=job_config)
load_job.result() # Wait for upload to finish
print("✅ Staging table populated.")

# Run an optimized MERGE statement to update the core table in-place
print("🔄 Executing SQL MERGE to update fact_sessions...")
merge_query = f"""
    MERGE `{PROJECT_ID}.patient_insights.fact_sessions` T
    USING `{PROJECT_ID}.patient_insights.tmp_cleaned_transcripts` S
    ON T.session_id = S.session_id
    WHEN MATCHED THEN
      UPDATE SET T.raw_transcript_unredacted = S.cleaned_transcript;
"""

bq_client.query(merge_query).result()

# Clean up the staging table
bq_client.delete_table(staging_table_id, not_found_ok=True)

print("🎉 Success! Your 12,000 sessions are cleaned and fully updated in BigQuery.")

🔍 Fetching sessions and patient metadata from BigQuery...
✅ Loaded 12000 sessions for cleaning.
🧹 Applying data-cleaning transformations...
💾 Uploading cleaned text to a temporary staging table...
✅ Staging table populated.
🔄 Executing SQL MERGE to update fact_sessions...
🎉 Success! Your 12,000 sessions are cleaned and fully updated in BigQuery.


In [ ]:
# 1. Add the files (using the correct uppercase name)
!git add data_dictionary.md  generation_checkpoint.json	README.md

# 2. Commit with corrected syntax
!git commit -m "Add project documentation"

# 3. Push
!git push https://:@github.com/meghansn/mental-health-llm-pipeline.git main

In [ ]:
from IPython.display import clear_output
# This will clear the output of all cells as it runs
for i in range(100):
    clear_output()